## MODEL OPTIMIZATION# 06 · Model Optimization

**Goal:** improve the CNN from notebook 04 by trying a few settings, then train a final model and save it.


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_DIR = Path("..")
IMAGES_DIR = PROJECT_DIR / "images"

TRAIN_DIR = PROJECT_DIR / "dataset" / "Training"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

In [3]:
class_names = sorted(folder.name for folder in TRAIN_DIR.iterdir()if folder.is_dir())
class_to_label = {name: i for i, name in enumerate(class_names)}

paths, labels = [], []

for class_name in class_names:
    for path in (TRAIN_DIR / class_name).rglob("*"):
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            paths.append(str(path))
            labels.append(class_to_label[class_name])

X_train, X_val, y_train, y_val = train_test_split(np.array(paths),np.array(labels),test_size=0.20,random_state=42,stratify=labels)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))

Training images: 4480
Validation images: 1120


In [4]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(image,channels=1,expand_animations=False)
    image.set_shape([None, None, 1])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.ensure_shape(image, [224, 224, 1])

    return image, label

In [5]:
def create_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(preprocess_image,num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        dataset = dataset.shuffle(buffer_size=len(paths),seed=42)

    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = create_dataset(X_train, y_train, training=True)
val_dataset = create_dataset(X_val, y_val)

## 1 Hyperparameter tuning

In [6]:
def build_cnn(filters1, filters2, dense_units):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 1)),
        layers.Conv2D(filters1, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(filters2, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dense(4, activation="softmax")])

    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

    return model

In [7]:
experiments = {"CNN_small": (16, 32, 64),
               "CNN_baseline": (32, 64, 128),
               "CNN_large": (64, 128, 128)}

In [ ]:
results = []
for name, params in experiments.items():
    print("\nTraining:", name)
    tf.keras.backend.clear_session()
    model = build_cnn(*params)
    history = model.fit(train_dataset,validation_data=val_dataset,epochs=10,verbose=1)

    results.append({"Model": name,"Best Validation Accuracy": max(history.history["val_accuracy"]),"Best Validation Loss": min(history.history["val_loss"])})


Training: CNN_small

Epoch 1/10


d:\MRI-Brain-Tumor-Classification\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


280/280 ━━━━━━━━━━━━━━━━━━━━ 101s 332ms/step - accuracy: 0.7362 - loss: 0.6687 - val_accuracy: 0.8223 - val_loss: 0.4667
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 76s 265ms/step - accuracy: 0.8842 - loss: 0.3045 - val_accuracy: 0.8661 - val_loss: 0.3701
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 57s 199ms/step - accuracy: 0.9402 - loss: 0.1602 - val_accuracy: 0.8929 - val_loss: 0.3051
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 75s 262ms/step - accuracy: 0.9699 - loss: 0.0903 - val_accuracy: 0.8902 - val_loss: 0.3551
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 251ms/step - accuracy: 0.9855 - loss: 0.0409 - val_accuracy: 0.8938 - val_loss: 0.4122
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 53s 185ms/step - accuracy: 0.9844 - loss: 0.0536 - val_accuracy: 0.8929 - val_loss: 0.4282
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 156s 552ms/step - accuracy: 0.9964 - loss: 0.0139 - val_accuracy: 0.8813 - val_loss: 0.4522
Epoch 8/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 46s 157ms/step - accuracy: 0.9980 - loss: 0.0067 - v

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.sort_values(by="Best Validation Accuracy",ascending=False))

          Model  Best Validation Accuracy  Best Validation Loss
1  CNN_baseline                  0.926786              0.267405
2     CNN_large                  0.926786              0.277657
0     CNN_small                  0.921429              0.298905


## 2 Learning-rate experiments

In [ ]:
def build_cnn_with_lr(learning_rate):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 1)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(4, activation="softmax")])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="sparse_categorical_crossentropy",metrics=["accuracy"])

    return model

In [ ]:
learning_rates = [0.01, 0.001, 0.0001]
lr_results = []

for lr in learning_rates:
    print(f"\nTraining CNN with learning rate: {lr}")
    tf.keras.backend.clear_session()
    model = build_cnn_with_lr(lr)

    history = model.fit(train_dataset,validation_data=val_dataset,epochs=5,verbose=1)

    lr_results.append({"Learning Rate": lr,
        "Best Validation Accuracy": max(history.history["val_accuracy"]),
        "Best Validation Loss": min(history.history["val_loss"])})


Training CNN with learning rate: 0.01
Epoch 1/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 155s 540ms/step - accuracy: 0.2451 - loss: 2.3881 - val_accuracy: 0.2500 - val_loss: 1.3876
Epoch 2/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 143s 466ms/step - accuracy: 0.2400 - loss: 1.3877 - val_accuracy: 0.2500 - val_loss: 1.3867
Epoch 3/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 153s 539ms/step - accuracy: 0.2384 - loss: 1.3876 - val_accuracy: 0.2500 - val_loss: 1.3864
Epoch 4/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 107s 378ms/step - accuracy: 0.2522 - loss: 1.3874 - val_accuracy: 0.2500 - val_loss: 1.3867
Epoch 5/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 106s 376ms/step - accuracy: 0.2335 - loss: 1.3876 - val_accuracy: 0.2500 - val_loss: 1.3866

Training CNN with learning rate: 0.001
Epoch 1/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 104s 363ms/step - accuracy: 0.7424 - loss: 0.6487 - val_accuracy: 0.7911 - val_loss: 0.4976
Epoch 2/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 114s 402ms/step - accuracy: 0.9165 - loss: 0.2221 - val_accuracy: 0.8911 - val_loss: 0.2838
Epoc

In [ ]:
lr_results

[{'Learning Rate': 0.01,
  'Best Validation Accuracy': 0.25,
  'Best Validation Loss': 1.3864461183547974},
 {'Learning Rate': 0.001,
  'Best Validation Accuracy': 0.9267857074737549,
  'Best Validation Loss': 0.28135037422180176},
 {'Learning Rate': 0.0001,
  'Best Validation Accuracy': 0.8928571343421936,
  'Best Validation Loss': 0.295421838760376}]

## 3 Dropout and regularization 

In [ ]:
def build_cnn_with_dropout(dropout_rate):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 1)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(4, activation="softmax")])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss="sparse_categorical_crossentropy",metrics=["accuracy"])

    return model

In [ ]:
dropout_rates = [0.0, 0.3, 0.5]
dropout_results = [] 

for rate in dropout_rates:
    print(f"\nTraining CNN with dropout rate: {rate}")
    tf.keras.backend.clear_session()
    model = build_cnn_with_dropout(rate)
    history = model.fit(train_dataset,validation_data=val_dataset,epochs=5,verbose=1)

    dropout_results.append({
        "Dropout Rate": rate,
        "Best Validation Accuracy": max(history.history["val_accuracy"]),
        "Best Validation Loss": min(history.history["val_loss"])})


Training CNN with dropout rate: 0.0
Epoch 1/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 114s 401ms/step - accuracy: 0.7576 - loss: 0.6564 - val_accuracy: 0.8661 - val_loss: 0.3643
Epoch 2/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 96s 339ms/step - accuracy: 0.9223 - loss: 0.2099 - val_accuracy: 0.9080 - val_loss: 0.2724
Epoch 3/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 97s 341ms/step - accuracy: 0.9645 - loss: 0.0990 - val_accuracy: 0.9000 - val_loss: 0.2896
Epoch 4/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 99s 349ms/step - accuracy: 0.9786 - loss: 0.0593 - val_accuracy: 0.9205 - val_loss: 0.3086
Epoch 5/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 101s 353ms/step - accuracy: 0.9915 - loss: 0.0271 - val_accuracy: 0.9179 - val_loss: 0.3058

Training CNN with dropout rate: 0.3
Epoch 1/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 108s 377ms/step - accuracy: 0.7199 - loss: 0.7412 - val_accuracy: 0.8250 - val_loss: 0.4280
Epoch 2/5
280/280 ━━━━━━━━━━━━━━━━━━━━ 103s 362ms/step - accuracy: 0.8815 - loss: 0.3066 - val_accuracy: 0.8821 - val_loss: 0.3034
Epoch 3/5
28

In [ ]:
dropout_results

[{'Dropout Rate': 0.0,
  'Best Validation Accuracy': 0.9205357432365417,
  'Best Validation Loss': 0.27239975333213806},
 {'Dropout Rate': 0.3,
  'Best Validation Accuracy': 0.9098214507102966,
  'Best Validation Loss': 0.26622501015663147},
 {'Dropout Rate': 0.5,
  'Best Validation Accuracy': 0.9196428656578064,
  'Best Validation Loss': 0.26090267300605774}]

In [ ]:
dropout_comparison = pd.DataFrame([
    {"Dropout Rate": 0.0,
     "Training Accuracy": 0.9915,
     "Validation Accuracy": 0.9205},
    {"Dropout Rate": 0.3,
     "Training Accuracy": 0.9770,
     "Validation Accuracy": 0.9098},
    {"Dropout Rate": 0.5,
     "Training Accuracy": 0.9516,
     "Validation Accuracy": 0.9196}])

print(dropout_comparison)

   Dropout Rate  Training Accuracy  Validation Accuracy
0           0.0             0.9915               0.9205
1           0.3             0.9770               0.9098
2           0.5             0.9516               0.9196


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(dropout_comparison["Dropout Rate"],dropout_comparison["Training Accuracy"],marker="o",label="Training Accuracy")
plt.plot(dropout_comparison["Dropout Rate"],[0.9179, 0.9089, 0.9196],marker="o",label="Validation Accuracy")
plt.xlabel("Dropout Rate")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(IMAGES_DIR / "06_dropout_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

NameError: name 'dropout_comparison' is not defined

<Figure size 800x500 with 0 Axes>

In [ ]:
loss_comparison = pd.DataFrame({
"Dropout Rate": [0.0, 0.3, 0.5],
"Training Loss": [0.0271, 0.0655, 0.1378],
"Validation Loss": [0.3058, 0.3416, 0.2609]})

print(loss_comparison)

   Dropout Rate  Training Loss  Validation Loss
0           0.0         0.0271           0.3058
1           0.3         0.0655           0.3416
2           0.5         0.1378           0.2609


In [ ]:
final_cnn_model = models.Sequential([
    layers.Input(shape=(224, 224, 1)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(4, activation="softmax")])

final_cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",metrics=["accuracy"])

final_cnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 200704)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    25,690,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,709,572 (98.07 MB)

 Trainable params: 25,709,572 (98.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
final_history = final_cnn_model.fit(train_dataset,validation_data=val_dataset,epochs=10,verbose=1)

Epoch 1/10


d:\MRI-Brain-Tumor-Classification\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


280/280 ━━━━━━━━━━━━━━━━━━━━ 96s 333ms/step - accuracy: 0.6799 - loss: 0.7978 - val_accuracy: 0.8330 - val_loss: 0.4729
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 94s 331ms/step - accuracy: 0.8359 - loss: 0.4385 - val_accuracy: 0.8536 - val_loss: 0.3797
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 1714s 6s/step - accuracy: 0.8819 - loss: 0.3063 - val_accuracy: 0.8839 - val_loss: 0.2977
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 105s 370ms/step - accuracy: 0.9085 - loss: 0.2348 - val_accuracy: 0.9134 - val_loss: 0.2372
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 101s 356ms/step - accuracy: 0.9357 - loss: 0.1725 - val_accuracy: 0.8973 - val_loss: 0.2943
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 102s 359ms/step - accuracy: 0.9455 - loss: 0.1477 - val_accuracy: 0.9259 - val_loss: 0.2379
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 100s 353ms/step - accuracy: 0.9545 - loss: 0.1213 - val_accuracy: 0.9214 - val_loss: 0.2760
Epoch 8/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 468s 2s/step - accuracy: 0.9676 - loss: 0.0875 - va

In [ ]:
final_cnn_model.save(PROJECT_DIR / "models" / "optimized_cnn_brain_tumor.keras")
print("Optimized CNN model saved successfully!")

Optimized CNN model saved successfully!


## Model Saving & Prediction.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("..")
MODELS_DIR = PROJECT_DIR / "models"

model_files = ["ann_brain_tumor.keras","cnn_brain_tumor.keras","optimized_cnn_brain_tumor.keras"]

for file in model_files:
    path = MODELS_DIR / file
    print(f"{file}: {'Saved' if path.exists() else 'Not found'}")

ann_brain_tumor.keras: Saved
cnn_brain_tumor.keras: Saved
optimized_cnn_brain_tumor.keras: Saved


In [ ]:
import tensorflow as tf
from pathlib import Path

PROJECT_DIR = Path("..")
MODELS_DIR = PROJECT_DIR / "models"

ann_model = tf.keras.models.load_model(MODELS_DIR / "ann_brain_tumor.keras")
cnn_model = tf.keras.models.load_model(MODELS_DIR / "cnn_brain_tumor.keras")
optimized_cnn_model = tf.keras.models.load_model(MODELS_DIR / "optimized_cnn_brain_tumor.keras")

print("All models loaded successfully!")

All models loaded successfully!


In [ ]:
print("ANN Input:", ann_model.input_shape)
print("ANN Output:", ann_model.output_shape)

print("\nCNN Input:", cnn_model.input_shape)
print("CNN Output:", cnn_model.output_shape)

print("\nOptimized CNN Input:", optimized_cnn_model.input_shape)
print("Optimized CNN Output:", optimized_cnn_model.output_shape)

ANN Input: (None, 50176)
ANN Output: (None, 4)

CNN Input: (None, 224, 224, 1)
CNN Output: (None, 4)

Optimized CNN Input: (None, 224, 224, 1)
Optimized CNN Output: (None, 4)


In [ ]:
import random
class_names = ["glioma", "meningioma", "notumor", "pituitary"]
selected_class = random.choice(class_names)
class_folder = PROJECT_DIR / "dataset" / "Testing" / selected_class
image_path = random.choice(list(class_folder.glob("*")))

print("Selected image:", image_path)
print("Actual class:", selected_class)

Selected image: ..\dataset\Testing\meningioma\Te-me_210.jpg
Actual class: meningioma


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.predict import predict_mri

result = predict_mri(image_path)

print("Predicted Class:", result["predicted_class"])
print("Confidence:", result["confidence"], "%")

print("\nClass Probabilities:")

for class_name, probability in result["probabilities"].items():
    print(f"{class_name}: {probability}%")

Image shape: (1, 224, 224, 1)
Minimum pixel value: 0.0
Maximum pixel value: 1.0


## 5 Final evaluation on the Testing folder

So far the optimized CNN was only checked on validation images. Here we test it on the Testing folder, exactly like in notebook 05, so we can compare it with the baseline CNN (83.1% test accuracy). **This is the number to report.**


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

TEST_DIR = PROJECT_DIR / "dataset" / "Testing"

test_paths, test_labels = [], []
for class_name in class_names:
    for path in (TEST_DIR / class_name).rglob("*"):
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            test_paths.append(str(path))
            test_labels.append(class_to_label[class_name])
test_paths, test_labels = np.array(test_paths), np.array(test_labels)

test_dataset = create_dataset(test_paths, test_labels)

test_loss, test_accuracy = optimized_cnn_model.evaluate(test_dataset)
print(f"Optimized CNN test accuracy: {test_accuracy * 100:.2f}%")
print(f"Optimized CNN test loss:     {test_loss:.4f}")

test_predictions = np.argmax(optimized_cnn_model.predict(test_dataset), axis=1)
print()
print(classification_report(test_labels, test_predictions, target_names=class_names))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step
Actual Class: meningioma
Predicted Class: meningioma
Confidence: 94.04%


In [ ]:
cm = confusion_matrix(test_labels, test_predictions)

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap="Purples", colorbar=False)
plt.title("Optimized CNN Confusion Matrix (Testing folder)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "06_optimized_cnn_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

notumor = class_to_label["notumor"]
tumor_rows = [i for i in range(len(class_names)) if i != notumor]
print("Tumor images predicted as notumor:", cm[tumor_rows, notumor].sum(), "of", cm[tumor_rows].sum())

glioma: 0.00%
meningioma: 94.04%
notumor: 5.96%
pituitary: 0.00%


## Summary

| Item | Choice |
|---|---|
| Model size | Baseline: Conv(32) → Conv(64) → Dense(128) (the bigger model gave no gain) |
| Learning rate | 0.001 (0.01 failed, 0.0001 was slower) |
| Dropout | 0.5 (smallest gap between training and validation) |
| Training | 10 epochs, Adam, batch size 16 |
| Validation accuracy | about 93% (baseline CNN: about 90% at the last epoch) |
| Test accuracy | printed in section 5 |
| Saved model | `models/optimized_cnn_brain_tumor.keras` |

**Figures saved for the README** (in `../images/`): `06_dropout_comparison.png`, `06_optimized_cnn_confusion_matrix.png`

**Next:** clean up `src/` and `app/`, then write the README.